# Delhi SUHII: four-model comparison and ablation
All four learners receive identical observations, folds and feature variants.
This is a fixed-configuration development benchmark, not an exact paper
reproduction, independent validation or proof of novelty.


In [ ]:
from pathlib import Path
import sys, subprocess, json
import pandas as pd
from IPython.display import display
ROOT = Path(r"C:/digitalTwin")
# Optional: set to the extracted ZIP folder to review bundled Linux results.
BUNDLE_ROOT = None
RUN_BENCHMARK = False
RUN_CONTROLS = False
print("Kernel:", sys.executable)


## Run the benchmark (optional)
Set RUN_BENCHMARK=True above, rerun setup and this cell. 80 fits.
Otherwise review saved results below without training.


In [ ]:
if RUN_BENCHMARK:
    subprocess.run([sys.executable, "-u", str(ROOT / "scripts/24_novelty_benchmark.py"),
                    "--root", str(ROOT), "--threads", "2"], cwd=ROOT, check=True)
else:
    print("Benchmark training disabled.")


In [ ]:
if BUNDLE_ROOT is not None:
    run = Path(BUNDLE_ROOT) / "results/benchmark"
else:
    runs = sorted(p for p in (ROOT / "models/experiments").glob("novelty_benchmark_*")
                  if (p / "pooled_oof_metrics.csv").is_file()
                  and (p / "fold_preprocessing.json").is_file())
    run = runs[-1] if runs else None
if run is None:
    print("No completed benchmark found. Run it or set BUNDLE_ROOT.")
else:
    print("Run:", run)
    scores = pd.read_csv(run / "summary.csv")
    display(scores.pivot(index="model", columns="variant", values="rmse"))
    display(scores[["model", "variant", "r2", "hot_rmse", "cold_rmse", "bias"]])
    display(pd.read_csv(run / "paired_ablation.csv"))


## Exploratory paired block-bootstrap evidence
Fixed OOF predictions, no model refits; intervals are conditional and not a formal significance or novelty test.

In [ ]:
if run is not None:
    subprocess.run([sys.executable, str(ROOT / "scripts/25_benchmark_evidence.py"),
                    "--run", str(run)], cwd=ROOT, check=True)
    display(pd.read_csv(run / "block_bootstrap_comparisons.csv"))


## Optional contrast falsification controls
30 fits: 3-km context, context + contrast, and context + 12 duplicated context predictors. XGBoost column subsampling is fixed at .8 or 1.0. Not automatic tuning.

In [ ]:
if RUN_CONTROLS:
    subprocess.run([sys.executable, "-u", str(ROOT / "scripts/26_contrast_controls.py"),
                    "--root", str(ROOT), "--threads", "2"], cwd=ROOT, check=True)
else:
    print("Control training disabled.")


## Claim boundaries
Feature contrasts are deterministic transformations of existing inputs. They
can improve representation without adding new measured information.
Same-year validation imagery is available to context construction; targets are
not. The experiment addresses imagery-available spatial mapping, not future-year
forecasting or causal mitigation effects. These folds informed contrast selection.
See the bundled literature matrix for existing related methods.
